## Example 1: gevent functionality test 1

In [ ]:
import gevent
import time

# Example 1: Basic greenlet creation and execution
print("Example 1: Basic greenlet creation and execution")

try:
    def worker_function(name, duration):
        """Simple worker that simulates work"""
        print(f"  Worker {name} starting, will work for {duration}s")
        gevent.sleep(duration)  # Non-blocking sleep
        print(f"  Worker {name} finished after {duration}s")
        return f"Result from {name}"
    
    # Create greenlets
    greenlet1 = gevent.spawn(worker_function, "A", 1)
    greenlet2 = gevent.spawn(worker_function, "B", 2)
    greenlet3 = gevent.spawn(worker_function, "C", 0.5)
    
    print(f"✓ Created {3} greenlets")
    print(f"  Greenlet types: {type(greenlet1).__name__}")
    
    # Wait for all to complete
    start_time = time.time()
    gevent.joinall([greenlet1, greenlet2, greenlet3])
    total_time = time.time() - start_time
    
    print(f"✓ All workers completed in {total_time:.2f}s (concurrent execution)")
    print(f"  Results: {[g.value for g in [greenlet1, greenlet2, greenlet3]]}")
    
    print("✓ Basic greenlet functionality demonstrated")
    
except Exception as e:
    print(f"✗ Error: {e}")
    raise AssertionError(
        f"gevent example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 1 finished\n")

## Example 2: gevent functionality test 2

In [ ]:
import gevent
from gevent import socket
from gevent.server import StreamServer

# Example 2: Asynchronous socket operations
print("Example 2: Asynchronous socket operations")

try:
    # A local echo server on an ephemeral port: the cell never leaves this machine.
    def echo(conn, address):
        data = conn.recv(1024)
        conn.sendall(b"echo:" + data)
        conn.close()

    server = StreamServer(("127.0.0.1", 0), echo)
    server.start()
    open_port = server.server_port

    # A port that is free right now: bind it, read the number, release it.
    probe = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    probe.bind(("127.0.0.1", 0))
    closed_port = probe.getsockname()[1]
    probe.close()

    def check_port(port, timeout=1):
        """Return (port, is_open, reply) for a TCP connect to 127.0.0.1:port."""
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        sock.settimeout(timeout)
        try:
            if sock.connect_ex(("127.0.0.1", port)) != 0:
                return port, False, None
            sock.sendall(b"ping")
            return port, True, sock.recv(1024)
        finally:
            sock.close()

    print(f"✓ Echo server listening on 127.0.0.1:{open_port}; probing it and port {closed_port}")

    # Check both ports concurrently, one greenlet each
    greenlets = [gevent.spawn(check_port, port) for port in (open_port, closed_port)]
    gevent.joinall(greenlets, timeout=5, raise_error=True)
    results = {g.value[0]: g.value[1:] for g in greenlets}
    server.stop()

    print(f"✓ Port check completed: {results}")
    assert results[open_port] == (True, b"echo:ping"), results[open_port]
    assert results[closed_port] == (False, None), results[closed_port]

    print("✓ Asynchronous socket operations demonstrated")

except Exception as e:
    print(f"✗ Error: {e}")
    raise AssertionError(
        f"gevent example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 2 finished\n")

## Example 3: gevent functionality test 3

In [ ]:
import gevent
from gevent.queue import Queue

# Example 3: Producer-consumer pattern with gevent queues
print("Example 3: Producer-consumer pattern with gevent queues")

try:
    # Create a queue for communication
    task_queue = Queue(maxsize=10)
    result_queue = Queue()
    
    def producer(name, items):
        """Produces items and puts them in the queue"""
        for i in range(items):
            task = f"{name}-task-{i}"
            task_queue.put(task)
            print(f"  Producer {name} created: {task}")
            gevent.sleep(0.1)  # Simulate work
        print(f"  Producer {name} finished")
    
    def consumer(name):
        """Consumes items from the queue"""
        processed = 0
        while processed < 3:  # Each consumer processes 3 items
            try:
                task = task_queue.get(timeout=2)
                result = f"processed-{task}"
                result_queue.put(result)
                print(f"  Consumer {name} processed: {task}")
                processed += 1
                gevent.sleep(0.2)  # Simulate processing
            except gevent.queue.Empty:
                break
        print(f"  Consumer {name} finished ({processed} items)")
    
    print(f"✓ Queue created (maxsize: {task_queue.maxsize})")
    
    # Start producers and consumers
    producers = [gevent.spawn(producer, f"P{i}", 3) for i in range(2)]
    consumers = [gevent.spawn(consumer, f"C{i}") for i in range(2)]
    
    # Wait for all to complete
    gevent.joinall(producers + consumers)
    
    # Check results
    results = []
    while not result_queue.empty():
        results.append(result_queue.get())
    
    print(f"✓ Producer-consumer completed:")
    print(f"  Tasks in queue: {task_queue.qsize()}")
    print(f"  Results collected: {len(results)}")
    
    print("✓ Queue-based communication demonstrated")
    
except Exception as e:
    print(f"✗ Error: {e}")
    raise AssertionError(
        f"gevent example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 3 finished\n")

## Example 4: gevent functionality test 4

In [ ]:
import gevent
from gevent.event import Event
import random

# Example 4: Event-driven coordination between greenlets
print("Example 4: Event-driven coordination between greenlets")

try:
    # Create events for coordination
    start_event = Event()
    finish_event = Event()
    
    def waiter(name):
        """Greenlet that waits for an event"""
        print(f"  {name} waiting for start event...")
        start_event.wait()  # Block until event is set
        
        # Simulate some work
        work_time = random.uniform(0.5, 1.5)
        print(f"  {name} starting work ({work_time:.2f}s)")
        gevent.sleep(work_time)
        
        print(f"  {name} finished work")
        return f"Result from {name}"
    
    def coordinator():
        """Coordinates the other greenlets"""
        print("  Coordinator: Preparing...")
        gevent.sleep(1)  # Simulate preparation time
        
        print("  Coordinator: Setting start event")
        start_event.set()  # Release all waiters
        
        # Wait a bit then signal finish
        gevent.sleep(2)
        print("  Coordinator: Setting finish event")
        finish_event.set()
    
    # Create waiters and coordinator
    waiters = [gevent.spawn(waiter, f"Worker-{i}") for i in range(3)]
    coord = gevent.spawn(coordinator)
    
    print(f"✓ Created {len(waiters)} waiters and 1 coordinator")
    print(f"  Start event set: {start_event.is_set()}")
    
    # Wait for coordinator to finish
    coord.join()
    
    # Wait for all workers with timeout
    gevent.joinall(waiters, timeout=5)
    
    # Check results
    completed = [w for w in waiters if w.ready()]
    print(f"✓ Event coordination completed:")
    print(f"  Workers completed: {len(completed)}/{len(waiters)}")
    print(f"  Finish event set: {finish_event.is_set()}")
    
    print("✓ Event-driven coordination demonstrated")
    
except Exception as e:
    print(f"✗ Error: {e}")
    raise AssertionError(
        f"gevent example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 4 finished\n")

## Example 5: gevent functionality test 5

In [ ]:
import gevent
from gevent.pool import Pool
import time

# Example 5: Greenlet pools for managing concurrent tasks
print("Example 5: Greenlet pools for managing concurrent tasks")

try:
    def simulate_task(task_id, duration):
        """Simulate a task that takes some time"""
        print(f"  Task {task_id} started (duration: {duration:.2f}s)")
        gevent.sleep(duration)
        result = f"Task-{task_id}-completed"
        print(f"  Task {task_id} finished")
        return result
    
    # Create a pool with limited size
    pool = Pool(size=3)  # Only 3 concurrent greenlets
    print(f"✓ Created pool with size: {pool.size}")
    
    # Generate tasks with different durations
    tasks = [(i, 0.5 + (i % 3) * 0.3) for i in range(8)]
    print(f"✓ Generated {len(tasks)} tasks")
    
    # Submit tasks to pool
    start_time = time.time()
    results = []
    
    for task_id, duration in tasks:
        result = pool.spawn(simulate_task, task_id, duration)
        results.append(result)
    
    print(f"✓ Submitted {len(tasks)} tasks to pool")
    print(f"  Free slots: {pool.free_count()}")
    
    # Wait for all tasks to complete
    pool.join()
    total_time = time.time() - start_time
    
    # Collect results
    completed_results = [r.value for r in results if r.ready()]
    
    print(f"✓ Pool execution completed:")
    print(f"  Total time: {total_time:.2f}s")
    print(f"  Completed tasks: {len(completed_results)}")
    print(f"  Pool is empty: {len(pool) == 0}")
    print(f"  Sample results: {completed_results[:3]}")
    
    print("✓ Greenlet pool management demonstrated")
    
except Exception as e:
    print(f"✗ Error: {e}")
    raise AssertionError(
        f"gevent example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 5 finished\n")